In [11]:
from sympy import *
from IPython.display import display
import numpy as np

M = 1   # Floquet truncation order

t, theta, omega_p = symbols('t theta omega_p', real=True)
k, n              = symbols('k n', integer=True)
omega             = IndexedBase('omega')   # omega[q] = frequency of mode q

Zs0 = symbols('Zs0', complex=True)
Yg0 = symbols('Yg0', complex=True)

# Zs^(m) and Yg^(m) as Functions of a frequency argument
# e.g. Zs_m[0](omega[k-1])  displays as  Zs1(omega[k-1])
Zs_m = [Function(f'Zs{m}') for m in range(1, M+1)]
Yg_m = [Function(f'Yg{m}') for m in range(1, M+1)]

V  = IndexedBase('V')
Ic = IndexedBase('I')

# Shorthand for the Fourier basis element
def E(j):
    return exp(I * j * omega_p * t)

xi = symbols('xi')   # placeholder frequency argument

def Zs_series(t):
    """Zs(t) with xi as placeholder frequency argument."""
    s = Zs0
    for mi, Zm in enumerate(Zs_m, start=1):
        s += Zm(xi) * E(mi) * exp( I*mi*theta) \
           + Zm(xi) * E(-mi) * exp(-I*mi*theta)
    return s

def Yg_series(t):
    """Yg(t) with xi as placeholder frequency argument."""
    s = Yg0
    for mi, Ym in enumerate(Yg_m, start=1):
        s += Ym(xi) * E(mi) * exp( I*mi*theta) \
           + Ym(xi) * E(-mi) * exp(-I*mi*theta)
    return s

Zs_t = Zs_series(t)
Yg_t = Yg_series(t)

Vn1_t = V[k, n]  - Zs_t * Ic[k, n]       # Zs(t) is the only time-varying thing
In1_t = Ic[k, n] - Yg_t * Vn1_t

# Expand so every term is a monomial in E(j) = exp(I*j*omega_p*t)
Vn1_expanded = expand(Vn1_t)
In1_expanded = expand(In1_t)


In [ ]:
js_nonzero = [j for j in range(-2*M, 2*M+1) if j != 0]
results = {}

# non-zero harmonics
for j_val in js_nonzero:
    cv_raw = Vn1_expanded.coeff(E(j_val))
    ci_raw = In1_expanded.coeff(E(j_val))
    cv = cv_raw.subs(xi, omega[k]).subs(k, k - j_val)
    ci = ci_raw.subs(xi, omega[k]).subs(k, k - j_val)
    results[j_val] = (cv, ci)

# j_val=0: subtract all oscillating terms from the full expression
# reconstructing the oscillating part in original (unshifted) k
V_osc_original = Add(*[Vn1_expanded.coeff(E(j)) * E(j) for j in js_nonzero])
I_osc_original = Add(*[In1_expanded.coeff(E(j)) * E(j) for j in js_nonzero])

cv0 = expand(Vn1_expanded - V_osc_original)   # pure DC, no E(j) left
ci0 = expand(In1_expanded - I_osc_original)

# xi shouldn't appear in DC terms, but substitute just in case
results[0] = (
    cv0.subs(xi, omega[k]),
    ci0.subs(xi, omega[k])
)

js = list(range(-2*M, 2*M+1))
V_full = Add(*[results[j_val][0] for j_val in js])
I_full = Add(*[results[j_val][1] for j_val in js])

# display(Eq(V[k, n+1], V_full))
# display(Eq(Ic[k, n+1], I_full))

Eq(V[k, n + 1], -Zs0*I[k, n] - Zs1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] - Zs1(omega[k - 1])*exp(I*theta)*I[k - 1, n] + V[k, n])

Eq(I[k, n + 1], Yg0*Zs0*I[k, n] + Yg0*Zs1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] + Yg0*Zs1(omega[k - 1])*exp(I*theta)*I[k - 1, n] - Yg0*V[k, n] + Zs0*Yg1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] + Zs0*Yg1(omega[k - 1])*exp(I*theta)*I[k - 1, n] - Yg1(omega[k + 1])*exp(-I*theta)*V[k + 1, n] + Yg1(omega[k + 2])*Zs1(omega[k + 2])*exp(-2*I*theta)*I[k + 2, n] - Yg1(omega[k - 1])*exp(I*theta)*V[k - 1, n] + Yg1(omega[k - 2])*Zs1(omega[k - 2])*exp(2*I*theta)*I[k - 2, n] + 2*Yg1(omega[k])*Zs1(omega[k])*I[k, n] + I[k, n])

In [14]:
# All state symbols that can appear: V[k+j, n] and Ic[k+j, n] for j in -M..M
state_syms = [V[k+j, n]  for j in range(-2*M, 2*M+1)] + \
             [Ic[k+j, n] for j in range(-2*M, 2*M+1)]

V_full_collected = collect(expand(V_full), state_syms)
I_full_collected = collect(expand(I_full), state_syms)

display(Eq(V[k, n+1],  V_full_collected))
display(Eq(Ic[k, n+1], I_full_collected))

Eq(V[k, n + 1], -Zs0*I[k, n] - Zs1(omega[k + 1])*exp(-I*theta)*I[k + 1, n] - Zs1(omega[k - 1])*exp(I*theta)*I[k - 1, n] + V[k, n])

Eq(I[k, n + 1], -Yg0*V[k, n] + (Yg0*Zs1(omega[k + 1])*exp(-I*theta) + Zs0*Yg1(omega[k + 1])*exp(-I*theta))*I[k + 1, n] + (Yg0*Zs1(omega[k - 1])*exp(I*theta) + Zs0*Yg1(omega[k - 1])*exp(I*theta))*I[k - 1, n] + (Yg0*Zs0 + 2*Yg1(omega[k])*Zs1(omega[k]) + 1)*I[k, n] - Yg1(omega[k + 1])*exp(-I*theta)*V[k + 1, n] + Yg1(omega[k + 2])*Zs1(omega[k + 2])*exp(-2*I*theta)*I[k + 2, n] - Yg1(omega[k - 1])*exp(I*theta)*V[k - 1, n] + Yg1(omega[k - 2])*Zs1(omega[k - 2])*exp(2*I*theta)*I[k - 2, n])

In [48]:
# ═══════════════════════════════════════════════════════════════════
#  STATE VECTOR ORDERING
#  [V[k-N,n], I[k-N,n], V[k-N+1,n], I[k-N+1,n], ..., V[k+N,n], I[k+N,n]]
#  interleaved V,I per sideband — size 2*(2N+1)
# ═══════════════════════════════════════════════════════════════════
N = 1
js_state = list(range(0, N+1)) 
# js_state = [-1, 0, 2, 5]

state_syms = []
for j in js_state:
    state_syms += [V[j, n], Ic[j, n]]

dim = len(state_syms)   # = 2*(2N+1)
print(f"State vector dimension: {dim}  (N={N})\n")
print("State vector ordering:")
for idx, s in enumerate(state_syms):
    print(f"  [{idx}]  {s}")

# ═══════════════════════════════════════════════════════════════════
#  BUILD SYMBOLIC TRANSFER MATRIX
#  For target mode k+p (p in -N..N):
#    substitute k -> k+p in results to get equations for mode k+p,
#    then extract coefficients of each state symbol.
# ═══════════════════════════════════════════════════════════════════
js_full = list(range(-2*M, 2*M+1))

T_sym = zeros(dim, dim)   # SymPy matrix

for row_p_idx, p in enumerate(js_state):
    # Get V[k+p, n+1] and I[k+p, n+1] expressions by shifting k -> k+p
    cv_p = Add(*[results[j][0].subs(k, p) for j in js_full])
    ci_p = Add(*[results[j][1].subs(k, p) for j in js_full])
    cv_p = expand(cv_p)
    ci_p = expand(ci_p)

    for col_idx, s in enumerate(state_syms):
        T_sym[2*row_p_idx,   col_idx] = cv_p.coeff(s)
        T_sym[2*row_p_idx+1, col_idx] = ci_p.coeff(s)

print("\nSymbolic transfer matrix T:")
display(T_sym)

State vector dimension: 4  (N=1)

State vector ordering:
  [0]  V[0, n]
  [1]  I[0, n]
  [2]  V[1, n]
  [3]  I[1, n]

Symbolic transfer matrix T:


Matrix([
[                          1,                                                            -Zs0,                            0,                                      -Zs1(omega[1])*exp(-I*theta)],
[                       -Yg0,                     Yg0*Zs0 + 2*Yg1(omega[0])*Zs1(omega[0]) + 1, -Yg1(omega[1])*exp(-I*theta), Yg0*Zs1(omega[1])*exp(-I*theta) + Zs0*Yg1(omega[1])*exp(-I*theta)],
[                          0,                                     -Zs1(omega[0])*exp(I*theta),                            1,                                                              -Zs0],
[-Yg1(omega[0])*exp(I*theta), Yg0*Zs1(omega[0])*exp(I*theta) + Zs0*Yg1(omega[0])*exp(I*theta),                         -Yg0,                       Yg0*Zs0 + 2*Yg1(omega[1])*Zs1(omega[1]) + 1]])

In [49]:

# ═══════════════════════════════════════════════════════════════════
#  NUMERICAL SUBSTITUTION
#
#  omega[q] = omega_s + q * omega_p   (equally spaced modes)
#  Zs_m and Yg_m are functions of frequency — replace with your own.
# ═══════════════════════════════════════════════════════════════════

# --- Edit these ---------------------------------------------------
omega_s_val = 1.0          # base frequency
omega_p_val = 0.1          # pump / modulation frequency
theta_val   = np.pi / 4
Zs0_val     = 1.0 + 0.0j
Yg0_val     = 0.5 + 0.0j

def omega_val(q):
    """Carrier frequency of mode q."""
    return omega_s_val + q * omega_p_val

def Zs_num(m, freq):
    """Z_s^(m)(omega).  Replace with your data or analytic formula."""
    return {1: 0.1 + 0.05j, 2: 0.02 + 0.01j}.get(m, 0+0j)

def Yg_num(m, freq):
    """Y_g^(m)(omega).  Replace with your data or analytic formula."""
    return {1: 0.05 + 0.02j, 2: 0.01 + 0.005j}.get(m, 0+0j)
# ------------------------------------------------------------------

def build_numeric_matrix(k_val=0):
    """
    Evaluate T_sym at a given centre mode index k_val.
    Returns a (dim x dim) numpy complex matrix.
    """
    # Build substitution dict
    subs = {
        Zs0:    Zs0_val,
        Yg0:    Yg0_val,
        theta:  theta_val,
        omega_p: omega_p_val,
        k:      k_val,
    }
    # omega[q] for all q that can appear: k_val ± (N + 2M)
    for dq in range(-(N + 2*M), (N + 2*M) + 1):
        q = k_val + dq
        subs[omega[q]] = omega_val(q)

    # Zs_m(omega[q]) and Yg_m(omega[q])
    for dq in range(-(N + 2*M), (N + 2*M) + 1):
        q = k_val + dq
        freq = omega_val(q)
        for mi, Zm in enumerate(Zs_m, start=1):
            subs[Zm(omega[q])] = Zs_num(mi, freq)
        for mi, Ym in enumerate(Yg_m, start=1):
            subs[Ym(omega[q])] = Yg_num(mi, freq)

    # Substitute into symbolic matrix
    T_num = np.zeros((dim, dim), dtype=complex)
    for i in range(dim):
        for j in range(dim):
            entry = T_sym[i, j].subs(subs)
            T_num[i, j] = complex(entry)
    return T_num

T_num = build_numeric_matrix(k_val=0)

print(f"\nNumerical transfer matrix  [k_val=0,  shape {T_num.shape}]")
print("Matrix:\n",  np.round(T_num.real, 6))


Numerical transfer matrix  [k_val=0,  shape (4, 4)]
Matrix:
 [[ 1.       -1.        0.       -0.106066]
 [-0.5       1.508    -0.049497  0.10253 ]
 [ 0.       -0.035355  1.       -1.      ]
 [-0.021213  0.038891 -0.5       1.508   ]]
